# Model check on Google Colab (M3)

Checks that the segmentation models work correctly before any experiment is run:

1. installs the pinned model library (`transformers`),
2. runs **all** tests, including the slow ones that download and run the real models,
3. scores SegFormer-B0, SegFormer-B2 and Mask2Former on **50 NightCity val images** and saves example
   pictures (none / gamma / CLAHE / gamma+CLAHE),
4. if Cityscapes is available: the **sanity check** - SegFormer-B0 on all 500 clean Cityscapes val images at
   short side 512 and 1024 (should be about 65% mIoU or more),
5. copies everything to `My Drive/lowlight-seg/results/m3_check/`.

**Use a GPU:** *Runtime -> Change runtime type -> T4 GPU*, then *Runtime -> Run all*. It takes about 10-20 minutes.
Run `colab_data_setup.ipynb` first (once). When it finishes, **save the printed output of steps 4-6**
(copy it into `docs/PROGRESS.md`) for the M3 checkpoint.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/lowlight-seg")  # your private project folder
CODE_ZIP = DRIVE / "code" / "lowlight-code.zip"
DATA_DRIVE = DRIVE / "data"            # prepared dataset zips live here
RAW_DRIVE = DATA_DRIVE / "raw"         # browser-downloaded original zips go here
RESULTS_DRIVE = DRIVE / "results"
REPO = Path("/content/lowlight-seg")   # the code is unpacked here (fast local disk)

for folder in (DATA_DRIVE, RAW_DRIVE, RESULTS_DRIVE):
    folder.mkdir(parents=True, exist_ok=True)

if not CODE_ZIP.is_file():
    raise FileNotFoundError(f"Upload lowlight-code.zip to {CODE_ZIP.parent} first (see 'Before you start').")
print("Drive connected:", DRIVE)

## 2. Unpack the code and record the environment

In [ ]:
import json, platform, shutil, subprocess, sys, zipfile

if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True)
with zipfile.ZipFile(CODE_ZIP) as zf:
    zf.extractall(REPO)
    COMMIT = zf.comment.decode() or "unknown"  # pack_code.ps1 (git archive) stores the commit id here

# Install the project package (dependencies used here are already preinstalled on Colab).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)
sys.path.insert(0, str(REPO / "src"))  # make `import lowlight` work in this running notebook too

import logging
import cv2, numpy, yaml
import lowlight.data_prep as dp

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

env = {
    "code_commit": COMMIT,
    "python": platform.python_version(),
    "numpy": numpy.__version__,
    "opencv": cv2.__version__,
    "pyyaml": yaml.__version__,
}
(RESULTS_DRIVE / "colab_env_data_setup.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

## 3. Install the pinned model library

In [ ]:
TRANSFORMERS_VERSION = "5.23.0"  # the version pinned in requirements.txt
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers=={TRANSFORMERS_VERSION}"],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
check = subprocess.run([sys.executable, "-c", "import torch, transformers; "
                        "print('torch', torch.__version__, '| transformers', transformers.__version__, "
                        "'| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"],
                       capture_output=True, text=True)
print(check.stdout, check.stderr[-2000:])
if "GPU: NONE" in check.stdout:
    print("WARNING: no GPU. Use Runtime -> Change runtime type -> T4 GPU, then run all again.")

## 4. Unpack the prepared data

In [ ]:
for z in sorted(DATA_DRIVE.glob("*.zip")):
    marker = REPO / "data" / f".{z.stem}_unpacked"
    if not marker.exists():
        print("Unpacking", z.name)
        dp.unzip_to(z, REPO / "data")
        marker.touch()
print(subprocess.run([sys.executable, "-m", "lowlight.cli", "check-data"], cwd=REPO,
                     capture_output=True, text=True).stdout)

## 5. Run every test, including the slow model tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-m", "slow or not slow", "-q", "-rs", "-p", "no:cacheprovider"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])

## 6. Score the three models on NightCity val (50 images) and save example pictures

In [ ]:
def run(args):
    r = subprocess.run([sys.executable, "scripts/model_check.py", *args], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-6000:], r.stderr[-4000:])

if (REPO / "data" / "nightcity").is_dir():
    run(["--dataset", "nightcity", "--split", "val", "--limit", "50"])
else:
    print("NightCity is not unpacked: run colab_data_setup.ipynb first.")

## 7. Sanity check on clean Cityscapes val (only if Cityscapes was downloaded)

In [ ]:
if (REPO / "data" / "cityscapes").is_dir():
    run(["--dataset", "cityscapes", "--split", "val", "--models", "segformer_b0",
         "--short-side", "512", "1024", "--grids", "2"])
else:
    print("Cityscapes not available yet - sanity check skipped (it runs once you have the account).")

## 8. Copy the results to Drive

In [ ]:
src = REPO / "results" / "m3_check"
if src.is_dir():
    shutil.copytree(src, RESULTS_DRIVE / "m3_check", dirs_exist_ok=True)
    print("Copied to", RESULTS_DRIVE / "m3_check")
    print((src / "summary.csv").read_text())